# 11 - Code quality, testing and profiling

**Sources:** Bootcamp (PEP8 / naming / formatting, Refactoring, Unit testing, Python optimisation) and Summer School
*BonusSlides* (profiling, speed-up tips).

Order of work for any serious piece of code: **make it work -> make it right (tests, clean) -> make it fast (profile
first!)**. "Premature optimisation is the root of all evil" - D. Knuth.

Everything below runs inside a throw-away temp directory, so the notebook leaves no files behind.

In [1]:
import subprocess, sys, tempfile, textwrap, os
from pathlib import Path

work = Path(tempfile.mkdtemp(prefix="refresher_"))

def sh(*args, cwd=work):
    """Run `python -m <tool> ...` with the same interpreter as the notebook and print the output."""
    r = subprocess.run([sys.executable, "-m", *args], cwd=cwd, capture_output=True, text=True)
    print((r.stdout + r.stderr).strip() or "(no output)")
    return r.returncode

def write(name, code):
    (work / name).write_text(textwrap.dedent(code).lstrip("\n"), encoding="utf-8")

## 1. Style: PEP 8 and naming
Readability counts (PEP 20, *The Zen of Python*). Key conventions:

| thing | convention |
|---|---|
| modules, packages | `lower_case`, short |
| functions, variables, methods | `snake_case` |
| classes, exceptions | `CamelCase` (`ValueError`, `HTTPError`) |
| constants | `UPPER_CASE` |
| "internal" names | leading `_name`; name-mangled `__name`; avoid trailing-underscore except to dodge keywords (`class_`) |

* 4 spaces per indent, lines <= 79 (black uses 88), two blank lines between top-level defs, imports at top in 3 groups
  (stdlib / third party / local), one statement per line, spaces around operators, **no** space before `(` in calls.
* Compare to `None` with `is`, booleans with truthiness (`if items:` not `if len(items) > 0:` / `== True`).
* Docstrings for every public module/class/function (PEP 257), comments explain *why*, not *what*.
* Don't use mutable default arguments, bare `except:`, or `from x import *`.

In [2]:
import this   # noqa: F401  (prints the Zen of Python)

The Zen of Python, by Tim Peters

Beautiful is better than ugly.
Explicit is better than implicit.
Simple is better than complex.
Complex is better than complicated.
Flat is better than nested.
Sparse is better than dense.
Readability counts.
Special cases aren't special enough to break the rules.
Although practicality beats purity.
Errors should never pass silently.
Unless explicitly silenced.
In the face of ambiguity, refuse the temptation to guess.
There should be one-- and preferably only one --obvious way to do it.
Although that way may not be obvious at first unless you're Dutch.
Now is better than never.
Although never is often better than *right* now.
If the implementation is hard to explain, it's a bad idea.
If the implementation is easy to explain, it may be a good idea.
Namespaces are one honking great idea -- let's do more of those!


## 2. Automatic tooling
| job | tool |
|---|---|
| formatter | **black** (opinionated), **isort** (import order) |
| linter (style + simple bugs) | **flake8** (or **ruff**, much faster, replaces several tools) |
| static type checker | **mypy** (or pyright) |
| security linter | **bandit** |
| all in git commit hook | **pre-commit** |

In [3]:
write("messy.py", '''
    import os,sys
    import json
    def Compute_total(Items,tax = 0.21 ,verbose=False):
        total=0
        for i in range(len(Items)):
            total = total + Items[i]*(1+tax)
        if verbose == True: print("total",total)
        return total
    class my_cart:
        def __init__(self,items=[]):
            self.items=items
''')
print(open(work / "messy.py").read())

import os,sys
import json
def Compute_total(Items,tax = 0.21 ,verbose=False):
    total=0
    for i in range(len(Items)):
        total = total + Items[i]*(1+tax)
    if verbose == True: print("total",total)
    return total
class my_cart:
    def __init__(self,items=[]):
        self.items=items



In [4]:
print("--- flake8 ---")
sh("flake8", "messy.py")

--- flake8 ---


messy.py:1:1: F401 'os' imported but unused
messy.py:1:1: F401 'sys' imported but unused
messy.py:1:10: E401 multiple imports on one line
messy.py:1:10: E231 missing whitespace after ','
messy.py:2:1: F401 'json' imported but unused
messy.py:3:1: E302 expected 2 blank lines, found 0
messy.py:3:24: E231 missing whitespace after ','
messy.py:3:28: E251 unexpected spaces around keyword / parameter equals
messy.py:3:30: E251 unexpected spaces around keyword / parameter equals
messy.py:3:35: E203 whitespace before ','
messy.py:3:36: E231 missing whitespace after ','
messy.py:4:10: E225 missing whitespace around operator
messy.py:7:16: E712 comparison to True should be 'if cond is True:' or 'if cond:'
messy.py:7:23: E701 multiple statements on one line (colon)
messy.py:7:38: E231 missing whitespace after ','
messy.py:9:1: E302 expected 2 blank lines, found 0
messy.py:10:22: E231 missing whitespace after ','
messy.py:11:19: E225 missing whitespace around operator


1

In [5]:
print("--- isort + black (in place) ---")
sh("isort", "messy.py")
sh("black", "messy.py")
print((work / "messy.py").read_text())

--- isort + black (in place) ---


Fixing C:\Users\leire\AppData\Local\Temp\refresher_yj2svskq\messy.py


reformatted messy.py

All done! \u2728 \U0001f370 \u2728
1 file reformatted.
import json
import os
import sys


def Compute_total(Items, tax=0.21, verbose=False):
    total = 0
    for i in range(len(Items)):
        total = total + Items[i] * (1 + tax)
    if verbose == True:
        print("total", total)
    return total


class my_cart:
    def __init__(self, items=[]):
        self.items = items



Black fixed layout, but **cannot fix design**: the function still uses `range(len(...))`, `== True`, a mutable default
argument, a class named in `snake_case`, and an unused import. That is what the refactoring section is for.

In [6]:
print("--- flake8 again (remaining issues) ---")
sh("flake8", "--max-line-length=88", "messy.py")

--- flake8 again (remaining issues) ---


messy.py:1:1: F401 'json' imported but unused
messy.py:2:1: F401 'os' imported but unused
messy.py:3:1: F401 'sys' imported but unused
messy.py:10:16: E712 comparison to True should be 'if cond is True:' or 'if cond:'


1

### Type hints + mypy
Hints are ignored at runtime; mypy checks them statically.

In [7]:
write("typed.py", '''
    from __future__ import annotations


    def mean(values: list[float]) -> float:
        return sum(values) / len(values)


    mean([1, 2, 3])
    mean("abc")          # error: str is not list[float]
    x: int = "text"      # error
''')
sh("mypy", "typed.py")

typed.py:9: error: Argument 1 to "mean" has incompatible type "str"; expected "list[float]"  [arg-type]
typed.py:10: error: Incompatible types in assignment (expression has type "str", variable has type "int")  [assignment]
Found 2 errors in 1 file (checked 1 source file)


1

### bandit - catches common security smells

In [8]:
write("risky.py", '''
    import pickle, subprocess, hashlib

    def run(user_input, blob):
        subprocess.call("ls " + user_input, shell=True)     # command injection
        obj = pickle.loads(blob)                             # arbitrary code execution
        eval(user_input)                                     # arbitrary code execution
        return hashlib.md5(b"password").hexdigest(), obj     # weak hash
''')
sh("bandit", "-q", "risky.py")

Run started:2026-10-12 17:11:02.406294+00:00

Test results:
>> Issue: [B403:blacklist] Consider possible security implications associated with pickle module.
   Severity: Low   Confidence: High
   CWE: CWE-502 (https://cwe.mitre.org/data/definitions/502.html)
   More Info: https://bandit.readthedocs.io/en/1.9.4/blacklists/blacklist_imports.html#b403-import-pickle
   Location: .\risky.py:1:0
1	import pickle, subprocess, hashlib
2	
3	def run(user_input, blob):

--------------------------------------------------
>> Issue: [B602:subprocess_popen_with_shell_equals_true] subprocess call with shell=True identified, security issue.
   Severity: High   Confidence: High
   CWE: CWE-78 (https://cwe.mitre.org/data/definitions/78.html)
   More Info: https://bandit.readthedocs.io/en/1.9.4/plugins/b602_subprocess_popen_with_shell_equals_true.html
   Location: .\risky.py:4:4
3	def run(user_input, blob):
4	    subprocess.call("ls " + user_input, shell=True)     # command injection
5	    obj = pickle.lo

1

## 3. Refactoring
Changing the *structure* of code without changing its *behaviour* - which is why you need tests first.
Typical smells: long functions, duplicated code, deep nesting, magic numbers, flag arguments, primitive obsession,
global state, comments that explain bad code.

In [9]:
# BEFORE
def Compute_total(Items, tax=0.21, verbose=False):
    total = 0
    for i in range(len(Items)):
        total = total + Items[i] * (1 + tax)
    if verbose == True:
        print("total", total)
    return total

# AFTER: guard clauses, no index loop, constant, no flag argument, docstring + hints
VAT = 0.21

def total_with_tax(prices: list[float], tax: float = VAT) -> float:
    """Return the sum of ``prices`` including ``tax`` (a fraction, e.g. 0.21)."""
    return sum(prices) * (1 + tax)

assert abs(Compute_total([10, 20]) - total_with_tax([10, 20])) < 1e-12     # behaviour preserved

In [10]:
# Deeply nested -> guard clauses (early return)
def discount_before(user, amount):
    if user is not None:
        if user.get("active"):
            if amount > 100:
                return amount * 0.9
            else:
                return amount
        else:
            return amount
    else:
        return amount

def discount(user, amount):
    if not user or not user.get("active") or amount <= 100:
        return amount
    return amount * 0.9

cases = [(None, 200), ({"active": False}, 200), ({"active": True}, 50), ({"active": True}, 200)]
print([(discount_before(u, a), discount(u, a)) for u, a in cases])

[(200, 200), (200, 200), (50, 50), (180.0, 180.0)]


In [11]:
# Replace if/elif chains with a dict dispatch, "flag" parameters with separate functions, tuples with dataclasses
import operator

OPS = {"+": operator.add, "-": operator.sub, "*": operator.mul, "/": operator.truediv}

def calc(a, op, b):
    try:
        return OPS[op](a, b)
    except KeyError:
        raise ValueError(f"unknown operator {op!r}") from None

print(calc(6, "*", 7))

42


## 4. Unit testing
Test pyramid: many fast **unit** tests, fewer **integration** tests, few end-to-end tests. A good unit test is
*fast, isolated, repeatable, self-checking, and timely* (FIRST). Pattern: **Arrange - Act - Assert**.

### `unittest` (stdlib, xUnit style)

In [12]:
write("shop.py", '''
    def total_with_tax(prices, tax=0.21):
        """Return the sum of prices including tax."""
        if tax < 0:
            raise ValueError("tax must be >= 0")
        return round(sum(prices) * (1 + tax), 2)


    def first_duplicate(seq):
        seen = set()
        for x in seq:
            if x in seen:
                return x
            seen.add(x)
        return None
''')
write("test_shop_unittest.py", '''
    import unittest
    from shop import total_with_tax, first_duplicate


    class TestTotal(unittest.TestCase):
        def setUp(self):                         # runs before every test
            self.prices = [10, 20]

        def test_basic(self):
            self.assertEqual(total_with_tax(self.prices), 36.3)

        def test_zero_tax(self):
            self.assertAlmostEqual(total_with_tax(self.prices, 0), 30)

        def test_negative_tax_raises(self):
            with self.assertRaises(ValueError):
                total_with_tax(self.prices, -1)


    class TestDuplicate(unittest.TestCase):
        def test_none(self):
            self.assertIsNone(first_duplicate([1, 2, 3]))

        def test_found(self):
            self.assertEqual(first_duplicate([1, 2, 1, 2]), 1)


    if __name__ == "__main__":
        unittest.main()
''')
sh("unittest", "-v", "test_shop_unittest")

test_found (test_shop_unittest.TestDuplicate.test_found) ... ok
test_none (test_shop_unittest.TestDuplicate.test_none) ... ok
test_basic (test_shop_unittest.TestTotal.test_basic) ... ok
test_negative_tax_raises (test_shop_unittest.TestTotal.test_negative_tax_raises) ... ok
test_zero_tax (test_shop_unittest.TestTotal.test_zero_tax) ... ok

----------------------------------------------------------------------
Ran 5 tests in 0.000s

OK


0

### `pytest` (the de-facto standard) - plain `assert`, fixtures, parametrisation, markers

In [13]:
write("test_shop_pytest.py", '''
    import pytest
    from shop import total_with_tax, first_duplicate


    @pytest.fixture
    def prices():
        return [10, 20]


    def test_basic(prices):
        assert total_with_tax(prices) == pytest.approx(36.3)


    @pytest.mark.parametrize(
        "seq, expected",
        [([], None), ([1, 2, 3], None), ([1, 2, 1], 1), (list("abcb"), "b")],
    )
    def test_first_duplicate(seq, expected):
        assert first_duplicate(seq) == expected


    def test_negative_tax(prices):
        with pytest.raises(ValueError, match="tax"):
            total_with_tax(prices, -0.1)


    @pytest.mark.xfail(reason="known bug: unhashable items", raises=TypeError)
    def test_unhashable():
        first_duplicate([[1], [1]])
''')
sh("pytest", "-q", "test_shop_pytest.py", "-p", "no:cacheprovider")

......x                                                                  [100%]
6 passed, 1 xfailed in 0.07s


0

### Mocking: isolate the unit from slow / non-deterministic collaborators

In [14]:
write("weather.py", '''
    import json
    import urllib.request


    def fetch_temperature(city):
        with urllib.request.urlopen(f"https://api.example.com/{city}") as r:
            return json.load(r)["temp"]


    def is_freezing(city):
        return fetch_temperature(city) <= 0
''')
write("test_weather.py", '''
    from unittest import mock
    import weather


    def test_is_freezing_true():
        with mock.patch.object(weather, "fetch_temperature", return_value=-3) as fake:
            assert weather.is_freezing("Oslo") is True
            fake.assert_called_once_with("Oslo")


    def test_is_freezing_false(monkeypatch):
        monkeypatch.setattr(weather, "fetch_temperature", lambda city: 21)
        assert weather.is_freezing("Madrid") is False
''')
sh("pytest", "-q", "test_weather.py", "-p", "no:cacheprovider")

..                                                                       [100%]
2 passed in 0.06s


0

### Coverage
`pytest --cov=shop --cov-report=term-missing` shows which lines the tests never executed. 100 % coverage does not
mean the tests are good, but low coverage means *something* is untested.

In [15]:
sh("pytest", "-q", "test_shop_pytest.py", "--cov=shop", "--cov-report=term-missing", "-p", "no:cacheprovider")

......x                                                                  [100%]
=============================== tests coverage ================================
_______________ coverage: platform win32, python 3.12.3-final-0 _______________

Name      Stmts   Miss  Cover   Missing
---------------------------------------
shop.py      11      0   100%
---------------------------------------
TOTAL        11      0   100%
6 passed, 1 xfailed in 0.06s


0

**Doctest** - examples in docstrings that are executed as tests (great for documentation):

In [16]:
import doctest

def square(x):
    """Square a number.

    >>> square(3)
    9
    >>> square(-1.5)
    2.25
    """
    return x * x

print(doctest.testmod())

TestResults(failed=0, attempted=2)


**TDD loop:** red (write a failing test) -> green (simplest code that passes) -> refactor. **Property-based testing**
(`hypothesis`) generates many inputs for you - worth learning once the basics feel natural.

## 5. Measuring performance
**Rule:** measure, don't guess. Use the right granularity:

| question | tool |
|---|---|
| how long does this expression take? | `timeit` / `%timeit` |
| which *function* is the bottleneck? | `cProfile` + `pstats` (or `snakeviz`) |
| which *line* is the bottleneck? | `line_profiler` (`kernprof`, `%lprun`) |
| how much memory? | `tracemalloc`, `memory_profiler` |

In [17]:
import timeit, time

print(timeit.timeit("sum(range(1000))", number=2000))
print(min(timeit.repeat("sum(range(1000))", repeat=5, number=2000)))          # use min of several repeats

t0 = time.perf_counter()                                                      # manual timing: perf_counter, not time.time
sum(range(10**6))
print(f"{time.perf_counter() - t0:.4f}s")

0.014328899793326855
0.0137749002315104
0.0198s


In [18]:
# Which is faster? Always test your assumptions.
setup = "data = list(range(10_000)); s = set(data)"
for stmt in ["9_999 in data", "9_999 in s"]:
    print(f"{stmt:15} {min(timeit.repeat(stmt, setup, repeat=5, number=2000)):.5f}s")

9_999 in data   0.07378s
9_999 in s      0.00004s


In [19]:
# cProfile: where does the time go?
import cProfile, pstats, io

def slow_primes(n):
    return [p for p in range(2, n) if all(p % d for d in range(2, p))]

def pipeline():
    a = slow_primes(3000)
    b = sorted(a, reverse=True)
    return sum(b)

pr = cProfile.Profile()
pr.enable(); pipeline(); pr.disable()
buf = io.StringIO()
pstats.Stats(pr, stream=buf).sort_stats("cumulative").print_stats(6)
print("\n".join(buf.getvalue().splitlines()[:18]))

         605349 function calls in 0.091 seconds

   Ordered by: cumulative time
   List reduced from 25 to 6 due to restriction <6>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        2    0.000    0.000    0.091    0.045 C:\Users\leire\Projects\Python-Refresher\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3786(run_code)
        2    0.000    0.000    0.091    0.045 {built-in method builtins.exec}
        1    0.000    0.000    0.090    0.090 C:\Users\leire\AppData\Local\Temp\ipykernel_30012\2450013218.py:1(<module>)
        1    0.000    0.000    0.090    0.090 C:\Users\leire\AppData\Local\Temp\ipykernel_30012\2450013218.py:7(pipeline)
        1    0.002    0.002    0.090    0.090 C:\Users\leire\AppData\Local\Temp\ipykernel_30012\2450013218.py:4(slow_primes)
     2998    0.045    0.000    0.089    0.000 {built-in method builtins.all}




In [20]:
# line_profiler (a script run via kernprof, or API as below)
from line_profiler import LineProfiler

def process(n=20000):
    result = []
    for i in range(n):
        if i % 3 == 0:
            result.append(i ** 2)
    total = sum(result)
    s = ",".join(str(x) for x in result)
    return total, len(s)

lp = LineProfiler(); wrapped = lp(process); wrapped()
buf = io.StringIO(); lp.print_stats(stream=buf, output_unit=1e-3)
print("\n".join(line for line in buf.getvalue().splitlines() if line.strip())[:1800])

Timer unit: 0.001 s
Total time: 0.011636 s
File: C:\Users\leire\AppData\Local\Temp\ipykernel_30012\3945948702.py
Function: process at line 4
Line #      Hits         Time  Per Hit   % Time  Line Contents
     4                                           def process(n=20000):
     5         1          0.0      0.0      0.0      result = []
     6     20001          4.0      0.0     34.1      for i in range(n):
     7     20000          4.3      0.0     36.8          if i % 3 == 0:
     8      6667          1.5      0.0     12.5              result.append(i ** 2)
     9         1          0.1      0.1      1.0      total = sum(result)
    10         1          1.8      1.8     15.5      s = ",".join(str(x) for x in result)
    11         1          0.0      0.0      0.0      return total, len(s)


In [21]:
# Memory: tracemalloc
import tracemalloc

tracemalloc.start()
big = [str(i) * 5 for i in range(100_000)]
current, peak = tracemalloc.get_traced_memory()
tracemalloc.stop()
print(f"current {current / 1e6:.1f} MB, peak {peak / 1e6:.1f} MB")
del big

import sys
print(sys.getsizeof([0] * 1000), sys.getsizeof((0,) * 1000), sys.getsizeof(range(1000)))     # list vs tuple vs range (lazy)

current 7.3 MB, peak 7.4 MB
8056 8040 48


## 6. Speed-up tips (in order of effectiveness)
1. **Better algorithm / data structure** (complexity!): `set`/`dict` lookups O(1) vs list O(n); `collections.deque` for queues;
   `bisect` for sorted data; `heapq` for top-k; sort once, not repeatedly.
2. **Vectorise** with NumPy/pandas instead of Python loops (see notebook 06).
3. **Use built-ins and the stdlib** (written in C): `sum`, `min`, `sorted`, `itertools`, `str.join`, comprehensions.
4. **Avoid repeated work**: hoist invariants out of loops, cache (`functools.lru_cache`), memoise, compute lazily (generators).
5. **Reduce attribute/global lookups** in hot loops (bind to locals) - minor gain.
6. **Don't concatenate strings in loops** (`''.join(parts)`), don't `list.insert(0, x)` / `pop(0)` (O(n)).
7. Parallelism: `multiprocessing` / `concurrent.futures.ProcessPoolExecutor` for CPU-bound work (GIL!),
   `threading` / `asyncio` for I/O-bound.
8. Compile: **Numba** (`@njit`), **Cython**, or rewrite hot spots in C/Rust. PyPy for pure-Python code.

In [22]:
from functools import lru_cache

def fib_slow(n):
    return n if n < 2 else fib_slow(n - 1) + fib_slow(n - 2)

@lru_cache(maxsize=None)
def fib_fast(n):
    return n if n < 2 else fib_fast(n - 1) + fib_fast(n - 2)

print(timeit.timeit(lambda: fib_slow(22), number=3), timeit.timeit(lambda: fib_fast(22), number=3))

0.005158000160008669 1.3200100511312485e-05


In [23]:
import numpy as np
import random

n = 200_000
xs = [random.random() for _ in range(n)]
arr = np.array(xs)

def py_loop():
    out = []
    for x in xs:
        out.append(x * x + 1)
    return out

def comp():
    return [x * x + 1 for x in xs]

def vec():
    return arr * arr + 1

for f in (py_loop, comp, vec):
    print(f"{f.__name__:8} {min(timeit.repeat(f, repeat=3, number=5)) / 5 * 1e3:8.2f} ms")

py_loop     11.46 ms
comp        11.08 ms
vec          1.09 ms


In [24]:
# String building: += in a loop vs join; list.pop(0) vs deque.popleft()
from collections import deque

def concat(n=20000):
    s = ""
    for i in range(n):
        s += str(i)
    return s

def joined(n=20000):
    return "".join(str(i) for i in range(n))

def drain_list(n=30000):
    q = list(range(n))
    while q:
        q.pop(0)

def drain_deque(n=30000):
    q = deque(range(n))
    while q:
        q.popleft()

for f in (concat, joined, drain_list, drain_deque):
    print(f"{f.__name__:12} {min(timeit.repeat(f, repeat=3, number=1)) * 1e3:8.2f} ms")

concat           2.07 ms
joined           1.33 ms


drain_list     845.95 ms
drain_deque      0.77 ms


In [25]:
# Parallel CPU-bound work (needs `if __name__ == "__main__"` in scripts; in a notebook we use threads-vs-processes idea only)
import os
print("CPU cores available:", os.cpu_count())

CPU cores available: 16


## 7. Project hygiene checklist
* `pyproject.toml` configures black/isort/ruff/mypy/pytest in one place; `requirements.txt` (or `pyproject` deps) pins versions.
* One virtual environment per project; never `pip install` into the system Python.
* Small functions with one job, tested; `main()` guarded by `if __name__ == "__main__":`.
* Use `logging` rather than `print` in libraries; raise specific exceptions; validate inputs at boundaries.
* Type hints on public functions; docstrings; meaningful names; a README with install/run/test instructions.
* Commit small, commit often; `.gitignore` your venv, caches and data.

In [26]:
import shutil
shutil.rmtree(work, ignore_errors=True)
print("cleaned", not work.exists())

cleaned True


## Exercises

**A.** Take the "messy" snippet from section 2 and refactor it into a `Cart` class with `add(item)`, `total()` and a
proper default argument. Write `pytest` tests *first* (parametrised), then aim for 100 % coverage.

**B.** Write `is_palindrome(text)` that ignores case, spaces and punctuation, using TDD. Include a doctest.

**C.** Profile this deliberately slow function with `cProfile` and `line_profiler`, then make it >50x faster and prove
it with `timeit`:
```python
def count_common(a, b):
    return len([x for x in a if x in b])      # a, b are lists of 20 000 ints
```

**D.** Mock `time.sleep` and `random.random` to test a `retry(fn, attempts, backoff)` helper with exponential backoff
and jitter - without really waiting.

**E.** Set up `pre-commit` with black, isort and flake8 in a scratch git repo and watch it reject a badly formatted commit.

In [27]:
# Your turn